<a href="https://colab.research.google.com/github/Tydos/LLMs-from-scratch/blob/main/LLM_from_scratch_chap2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#setup
!pip install torch tiktoken tensorflow tqdm numpy pandas psutil


In [ ]:
from importlib_metadata import version
print(version('torch'))
print(version('tiktoken'))

2.10.0+cpu
0.12.0


In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt","r") as f:
    raw_text = f.read()

print(len(raw_text))
print(raw_text[:1000])

20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even thro

In [ ]:
#do some basic text splitting
import re
regex_pattern = r"\W+"
split_text = re.split(regex_pattern, raw_text)
split_text = [w.lower() for w in split_text if w]
print(f'Total tokens: {len(split_text)}')
print(split_text[:100])

Total tokens: 3826
['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', 'though', 'a', 'good', 'fellow', 'enough', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', 'in', 'the', 'height', 'of', 'his', 'glory', 'he', 'had', 'dropped', 'his', 'painting', 'married', 'a', 'rich', 'widow', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'riviera', 'though', 'i', 'rather', 'thought', 'it', 'would', 'have', 'been', 'rome', 'or', 'florence', 'the', 'height', 'of', 'his', 'glory', 'that', 'was', 'what', 'the', 'women', 'called', 'it', 'i', 'can', 'hear', 'mrs', 'gideon', 'thwing', 'his', 'last', 'chicago', 'sitter', 'deploring', 'his', 'unaccountable', 'abdication', 'of', 'course', 'it', 's', 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', 'way']


In [ ]:
#create a set of words
all_words: set[str] = sorted(set(split_text))
all_words.extend(['<unk>','<eos>'])
print(f'Total unique words: {len(all_words)}')
print(all_words[:100])

Total unique words: 1086
['_am_', '_famille', '_felt_', '_has_', '_have_', '_i', '_jardiniere_', '_mine_', '_not_', '_rose', '_rs_', '_that_', '_the_', '_was_', '_were_', 'a', 'abdication', 'able', 'about', 'above', 'abruptly', 'absolute', 'absorbed', 'absurdity', 'academic', 'accuse', 'accustomed', 'across', 'activity', 'add', 'added', 'admirers', 'adopted', 'adulation', 'advance', 'aesthetic', 'affect', 'afraid', 'after', 'afterward', 'again', 'ago', 'ah', 'air', 'alive', 'all', 'almost', 'alone', 'along', 'always', 'amazement', 'amid', 'among', 'amplest', 'amusing', 'an', 'and', 'another', 'answer', 'answered', 'any', 'anything', 'anywhere', 'apparent', 'apparently', 'appearance', 'appeared', 'appointed', 'are', 'arm', 'arms', 'arrt', 'art', 'articles', 'artist', 'as', 'aside', 'asked', 'at', 'atmosphere', 'atom', 'attack', 'attention', 'attitude', 'audacities', 'away', 'awful', 'axioms', 'azaleas', 'back', 'background', 'balance', 'balancing', 'balustraded', 'basking', 'bath', 'be'

In [ ]:
#create a vocab map word->int
vocab: dict[str|int] = {
    word:idx for idx,word in enumerate(all_words)
}
len(vocab)

1086

In [ ]:
from IPython.lib.display import Iterable
import re

class Tokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {v: k for k, v in vocab.items()}

    def encode(self, text):
        # Simple split on non-word characters
        tokens = re.split(r'(\W+)', text.lower())
        tokens = [t.strip() for t in tokens if t.strip()]
        return [self.str_to_int.get(t, self.str_to_int['<unk>']) for t in tokens]

    def decode(self, tokens:Iterable):
        # Simple join
        return " ".join([self.int_to_str.get(t, '<unk>') for t in tokens])

tk = Tokenizer(vocab)
# Using words that exist in 'the-verdict.txt'
text = "I had always thought."
ids = tk.encode(text)
print(f"IDs: {ids}")
print(f"Text: {tk.decode(ids)}")

IDs: [485, 435, 49, 949, 1084]
Text: i had always thought <unk>


In [ ]:
#convert entire data to tokens
tokenized_text =  tk.encode(raw_text)
print(tokenized_text[:100])

[485, 435, 49, 949, 511, 407, 755, 15, 161, 402, 1084, 948, 15, 419, 348, 306, 1084, 851, 508, 1029, 640, 426, 907, 964, 588, 964, 455, 933, 1084, 491, 934, 458, 654, 470, 415, 1084, 453, 435, 282, 470, 680, 1084, 586, 15, 779, 1055, 1084, 56, 311, 468, 491, 15, 1018, 659, 934, 782, 1084, 948, 485, 755, 949, 508, 1073, 450, 106, 785, 666, 361, 1084, 934, 458, 654, 470, 415, 1084, 933, 1029, 1043, 934, 1065, 144, 508, 1084, 485, 146, 455, 621, 1084, 406, 955, 1084, 470, 526, 165, 840, 1084, 238, 470, 991, 16]


In [ ]:
#mask the tokens model will not seeing since LLM generate one word at a time by seeing the previous words
for i in range(5):
  context = tokenized_text[:i]
  target = tokenized_text[i+1]
  print(f'Context: {tk.decode(context)} -> {tk.decode([target])}')


Context:  -> had
Context: i -> always
Context: i had -> thought
Context: i had always -> jack
Context: i had always thought -> gisburn


In [ ]:
#create the entire dataset using this logic, we use pytorch dataloader here
from torch.utils.data import Dataset, DataLoader
import torch

class GPTDataset(Dataset):
  #max len - amount of words that model sees as context
  #stride - commonality to ensure that model can form relationships
  def __init__(self,text,tokenizer,max_len,stride):
    self.input_ids = []
    self.target_ids = []
    self.tokenizer = tokenizer
    self.max_len = max_len
    self.stride = stride
    tokenized_text = self.tokenizer.encode(text)
    for i in range(0,len(tokenized_text)-max_len,stride):
      input_chunk = tokenized_text[i:i+max_len]
      target_chunk = tokenized_text[i+1:i+max_len+1]

      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return (self.input_ids[idx], self.target_ids[idx])

In [ ]:
#dataloader

def GPTLoader(text,tokenizer=tk,max_len=3,stride=1,batch_size=10):
  dataset = GPTDataset(text,tokenizer,max_len,stride)

  return DataLoader(dataset,batch_size=batch_size,shuffle=True)


In [ ]:
with open("the-verdict.txt","r") as f:
  raw_text = f.read()

dataloader = GPTLoader(raw_text,tokenizer=tk,max_len=3,stride=1,batch_size=1)

for data in dataloader:
  print(data)
  break

[tensor([[1084,  489,  485]]), tensor([[489, 485, 202]])]
